## Load data from Azure SQL Database using JDBC

In [0]:
# Insert your database credentials and URL
url = (
    "jdbc:sqlserver://jarvis-sql.database.windows.net:1433;"
    "database=jrvs-tic;"
    "encrypt=true;"
    "trustServerCertificate=false;"
    "hostNameInCertificate=*.database.windows.net;"
    "loginTimeout=30;"
)

In [0]:
%sql
-- set the catalog
USE jarvis_training_catalog.bronze;

In [0]:
bronze_users_df = (spark.read
  .format("jdbc")
  .option("url", url)
  .option("dbtable", "users_data")
  .option("user", "ahu-jrvs")
  .option("password", "PASSWORD")
  .load()
)

display(bronze_users_df.head(5)) # display your dataframe

bronze_users_df.write.mode('overwrite').saveAsTable('users')

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,33,69,1986,3,Male,858 Plum Avenue,43.59,-70.33,29237.000000000000000000,59613.000000000000000000,36199.000000000000000000,763,4
1,43,74,1976,4,Female,113 Burns Lane,30.44,-87.18,22247.000000000000000000,45360.000000000000000000,14587.000000000000000000,704,3
2,48,64,1971,8,Male,6035 Forest Avenue,40.84,-73.87,13461.000000000000000000,27447.000000000000000000,80850.000000000000000000,673,5
3,49,65,1970,12,Male,840 Elm Avenue,33.89,-98.51,13705.000000000000000000,27943.000000000000000000,18693.000000000000000000,681,4
4,54,72,1965,3,Female,6016 Little Creek Boulevard,47.61,-122.3,37485.000000000000000000,76431.000000000000000000,115362.000000000000000000,716,5


In [0]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, DateType, DecimalType,
    StringType, DoubleType
)
from pyspark.sql.functions import col

df = (spark.read
  .format("jdbc")
  .option("url", url)
  .option("dbtable", "transaction_data_cleaned")
  .option("user", "ahu-jrvs")
  .option("password", "PASSWORD")
  .load()
)

bronze_transactions_df = (
    df
    .withColumn("id", col("id").cast("int"))
    .withColumn("date", col("date").cast("date"))
    .withColumn("client_id", col("client_id").cast("int"))
    .withColumn("card_id", col("card_id").cast("int"))
    .withColumn("amount", col("amount").cast("decimal(20, 2)"))
    .withColumn("use_chip", col("use_chip").cast("string"))
    .withColumn("merchant_id", col("merchant_id").cast("int"))
    .withColumn("merchant_city", col("merchant_city").cast("string"))
    .withColumn("merchant_state", col("merchant_state").cast("string"))
    .withColumn("zip", col("zip").cast("double"))
    .withColumn("mcc", col("mcc").cast("int"))
    .withColumn("errors", col("errors").cast("string"))
)

# write to table
bronze_transactions_df.write.mode('overwrite').saveAsTable('transactions')

# display(bronze_transactions_df.head(5)) # display your dataframe

In [0]:
bronze_card_df = (spark.read
  .format("jdbc")
  .option("url", url)
  .option("dbtable", "cards_data")
  .option("user", "ahu-jrvs")
  .option("password", "PASSWORD")
  .load()
)

display(bronze_card_df.head(5)) # display your dataframe

bronze_card_df.write.mode('overwrite').saveAsTable('cards')

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,1362,Amex,Credit,393314135668401,04/2024,866,YES,2,33900.000000000000000000,01/1991,2014,No
1,550,Mastercard,Credit,5278231764792292,06/2024,396,YES,1,11600.000000000000000000,01/1994,2013,No
2,556,Mastercard,Debit,5889825928297675,09/2021,422,YES,1,19948.000000000000000000,01/1995,2011,No
3,1937,Visa,Credit,4289888672554714,04/2020,736,YES,2,16400.000000000000000000,01/1995,2015,No
4,1981,Mastercard,Debit,5433366978583845,03/2024,530,YES,2,19439.000000000000000000,01/1997,2007,No
